# Kafka Consumer

Using the `kafka-python` module, Kafka consumers can be instantiated via the `KafkaConsumer` class:

```python
from kafka import KafkaConsumer

# Create a Kafka consumer instance
consumer = KafkaConsumer(
    bootstrap_servers=['62.30.10.23:9092'],  # list of brokers
    security_protocol="SSL",                 # security protocol (if any) 
    ssl_cafile="./ca.pem",                   # certificate details (if any)
    ssl_certfile="./service.cert",           #           ...
    ssl_keyfile="./service.key",             #           ...
    value_deserializer=msgpack.unpackb,      # message value deserialization function (e.g. unpack the message from a specific format)
    auto_offset_reset='earliest',            # automatically bring the reading offset to the earliest message
    group_id="group_A",                      # identify this consumer as part of group_A
)
```

Once more, we'll use a simple implementation of the consumer, with no specific configurations used in this example.

In [1]:
# define the list of brokers in the cluster
KAFKA_BOOTSTRAP_SERVERS = ['kafka-broker:9092']

In [2]:
from kafka import KafkaConsumer

# create a Kafka consumer instance
consumer = KafkaConsumer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,  # list of Kafka brokers
    consumer_timeout_ms=10000                   # maximum time to wait for a new message 
                                                # before stopping the consumer
)

Inspect the available topics on the brokers:

In [3]:
# list all available topics on the kafka brokers
consumer.topics()

{'a_partitioned_topic', 'my_awesome_topic'}

In the Publish/Subscribe (Pub/Sub) model, before consuming messages from a specific topic, you need to subscribe to that topic. By subscribing, the consumer expresses its interest in receiving messages from the specified topic.

It's important to note that subscribing to a topic doesn't immediately consume any messages. 

Instead, it establishes a connection between the consumer and the partitions of the subscribed topic hosted on the Kafka brokers. This connection allows the consumer to start polling for messages from those partitions.

Once you have subscribed to the topic, the consumer can start polling for messages, fetching messages from the subscribed partitions and return them to the consumer for further processing.

In [4]:
# subscribe to a topic
consumer.subscribe(['my_awesome_topic'])

# check the active subscriptions
consumer.subscription()

{'my_awesome_topic'}

The `KafkaConsumer` class also offers the possibility to inspect topics (for instance, in terms of the number of partitions), but **not** to modify them. 

We can inspect how many partitions the specific topic is made of:

In [5]:
# print the list of partition IDs 
# e.g. a topic with tree partitions will have partition IDs {0, 1, 2}
consumer.partitions_for_topic('my_awesome_topic')

{0}

We can instruct the consumer to `poll` (i.e., ask for all new messages stored in the topic) with a given cadence or logic.

For instance, we can set the consumer to read only 10 messages at a time, with a timeout between subsequent readouts of a given $\Delta t$.

In [6]:
# set up the polling strategy for the consumer
consumer.poll(timeout_ms=0,         # do not enable dead-times before one poll to the next
              max_records=None,     # do not limit the number of records to consume at once 
              update_offsets=True   # update the reading offsets on this topic
             )

{}

Now the consumer is ready to poll messages (until it is stopped or it reaches a timeout).

Let's look for all messages in the consumer:

In [11]:
# this consumer will keep polling for messages 
# until stopped by the user
# (or reaches the consumer_timeout_ms, if specified)
for message in consumer:
    print (message)

The reading offset can also be reset to the beginning of the topic, allowing for re-reading the entire topic:

In [9]:
# go back to the beginning of the topic
consumer.seek_to_beginning()

# read the entire topic and print it out
for message in consumer:
    print (message)

ConsumerRecord(topic='my_awesome_topic', partition=0, leader_epoch=0, offset=0, timestamp=1748503210782, timestamp_type=0, key=None, value=b"let's try...", headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=12, serialized_header_size=-1)
ConsumerRecord(topic='my_awesome_topic', partition=0, leader_epoch=0, offset=1, timestamp=1748503222085, timestamp_type=0, key=None, value=b'is anything really happening?', headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=29, serialized_header_size=-1)
ConsumerRecord(topic='my_awesome_topic', partition=0, leader_epoch=0, offset=2, timestamp=1748503230518, timestamp_type=0, key=None, value=b"doesn't seem so", headers=[], checksum=None, serialized_key_size=-1, serialized_value_size=15, serialized_header_size=-1)
ConsumerRecord(topic='my_awesome_topic', partition=0, leader_epoch=0, offset=3, timestamp=1748503413034, timestamp_type=0, key=None, value=b'try now', headers=[], checksum=None, serialized_key_size=-1

The message content (`ConsumerRecord`) can be quite messy, but it can be easily inspected by parsing only the relevant information.

In [10]:
# import packages to interpret dates
from datetime import datetime

# go back to the beginning of the topic
consumer.seek_to_beginning()

# break down the message into its main components
for message in consumer:
    print ("%d[%d] @%s k=%s v=%s" % (message.partition,
                          message.offset,
                          datetime.fromtimestamp(message.timestamp/1000).time(),
                          message.key,
                          message.value))

0[0] @07:20:10.782000 k=None v=b"let's try..."
0[1] @07:20:22.085000 k=None v=b'is anything really happening?'
0[2] @07:20:30.518000 k=None v=b"doesn't seem so"
0[3] @07:23:33.034000 k=None v=b'try now'
0[4] @07:23:56.757000 k=None v=b'yay'
0[5] @07:30:26.879000 k=None v=b'message 1'
0[6] @07:32:07.097000 k=None v=b'a new message'
0[7] @07:33:03.001000 k=None v=b'a new message'
0[8] @07:33:42.715000 k=None v=b'a message from a newly revived producer'
0[9] @07:34:56.296000 k=b'some_key' v=b'a message with key'
0[10] @07:45:23.599000 k=None v=b'hi'
0[11] @07:45:30.955000 k=None v=b'im angela'
0[12] @07:45:39.872000 k=None v=b'imadding messages'
0[13] @07:45:49.238000 k=None v=b'it continues working'
0[14] @07:46:00.750000 k=None v=b'till i reach the 10s timeout'


Let's change the topic to which the consumer is subscribed and make it a partitioned one:

In [12]:
# subscribe to a partitioned topic
consumer.subscribe(['a_partitioned_topic'])
consumer.subscription()

{'a_partitioned_topic'}

By inspecting the number of partitions for this topic, we can now see that there are 2 partitions: partition #0 and partition #1.

In [14]:
# check the partitions in the partitioned topic
consumer.partitions_for_topic('a_partitioned_topic')

{0, 1}

When reading from a partitioned topic, it's easy to observe that the messages are sent to the two partitions in a seemingly arbitrary way.

In [15]:
# import json to unpack json strings
import json

# go back to the beginning of the topic
consumer.seek_to_beginning()

# read messages from the beginning of the topic
# decode the json into the python dictionary format
for message in consumer:
    print(f"{message.partition}[{message.offset}]\t {json.loads(message.value)}")

0[0]	 {'name': 'John', 'surname': 'Smith', 'amount': 90.04, 'delta_t': 7.16, 'flag': 0}
0[1]	 {'name': 'Alice', 'surname': 'Smith', 'amount': 94.08, 'delta_t': 9.02, 'flag': 0}
0[2]	 {'name': 'Joe', 'surname': 'Smith', 'amount': 195.89, 'delta_t': 6.38, 'flag': 0}
0[3]	 {'name': 'John', 'surname': 'Smith', 'amount': 328.82, 'delta_t': 8.1, 'flag': 0}
0[4]	 {'name': 'Andy', 'surname': 'Smith', 'amount': 499.96, 'delta_t': 1.83, 'flag': 0}
0[5]	 {'name': 'John', 'surname': 'Smith', 'amount': 594.69, 'delta_t': 4.5, 'flag': 1}
0[6]	 {'name': 'Alice', 'surname': 'Jones', 'amount': 81.74, 'delta_t': 2.92, 'flag': 0}
0[7]	 {'name': 'John', 'surname': 'Smith', 'amount': 367.85, 'delta_t': 6.43, 'flag': 0}
0[8]	 {'name': 'Alice', 'surname': 'Millers', 'amount': 215.14, 'delta_t': 9.61, 'flag': 0}
0[9]	 {'name': 'Andy', 'surname': 'Millers', 'amount': 155.91, 'delta_t': 2.92, 'flag': 0}
0[10]	 {'name': 'Alice', 'surname': 'Jones', 'amount': 249.23, 'delta_t': 6.73, 'flag': 0}
0[11]	 {'name': 'A

each partition (first column) has its own offset

## Creating a Consumer Accessing Only One Partition

Publishing records to a partitioned topic is typically transparent to the user: the producer publishes to the topic, and the Kafka cluster will redirect the message to the partition leader, later replicating it to the followers.

The same applies to a generic consumer. As we have just seen, data is consumed from all partitions within the topic.

In some cases, however, it may be more suitable to instantiate multiple consumers, with each one reading from a specific partition of a topic.

Let's create a consumer specifically designed to access the data from partition #0 of the previous partitioned topic.

In [16]:
# import TopicPartition to be able to assign a partition to a consumer
from kafka import TopicPartition

# create a standard consumer with a timeout of 10 seconds
consumer_part_0 = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
                                client_id='consumer_n_0',
                                consumer_timeout_ms=10000)

# assign the consumer to a specific topic-partition combination
consumer_part_0.assign([TopicPartition(topic     = 'a_partitioned_topic', # topic
                                       partition = 0                      # partition id
                                      )]
                      ) 

In [17]:
# go back to the beginning of the topic (for this specific partition)
consumer_part_0.seek_to_beginning()

# read messages from the beginning of the topic (for this specific partition)
# decode the json into the python dictionary format
for message in consumer_part_0:
    print(f"{message.partition}[{message.offset}]\t {json.loads(message.value)}")

0[0]	 {'name': 'John', 'surname': 'Smith', 'amount': 90.04, 'delta_t': 7.16, 'flag': 0}
0[1]	 {'name': 'Alice', 'surname': 'Smith', 'amount': 94.08, 'delta_t': 9.02, 'flag': 0}
0[2]	 {'name': 'Joe', 'surname': 'Smith', 'amount': 195.89, 'delta_t': 6.38, 'flag': 0}
0[3]	 {'name': 'John', 'surname': 'Smith', 'amount': 328.82, 'delta_t': 8.1, 'flag': 0}
0[4]	 {'name': 'Andy', 'surname': 'Smith', 'amount': 499.96, 'delta_t': 1.83, 'flag': 0}
0[5]	 {'name': 'John', 'surname': 'Smith', 'amount': 594.69, 'delta_t': 4.5, 'flag': 1}
0[6]	 {'name': 'Alice', 'surname': 'Jones', 'amount': 81.74, 'delta_t': 2.92, 'flag': 0}
0[7]	 {'name': 'John', 'surname': 'Smith', 'amount': 367.85, 'delta_t': 6.43, 'flag': 0}
0[8]	 {'name': 'Alice', 'surname': 'Millers', 'amount': 215.14, 'delta_t': 9.61, 'flag': 0}
0[9]	 {'name': 'Andy', 'surname': 'Millers', 'amount': 155.91, 'delta_t': 2.92, 'flag': 0}
0[10]	 {'name': 'Alice', 'surname': 'Jones', 'amount': 249.23, 'delta_t': 6.73, 'flag': 0}
0[11]	 {'name': 'A

## Creating a Consumer Group

Multiple consumers can read from the same topic.

In Kafka, every consumer is part of a consumer group (even a single consumer is part of its own consumer group).

A consumer group consists of one or more cooperating consumers that gather data from the same topic. The consumer group dynamically balances the load across its members and redistributes the consume calls.

If a consumer within a consumer group fails, the remaining consumers in the same group will continue to read the entire data from the subscribed topic.

In [18]:
# create `consumer_one` to read from a specific partition
# assign this consumer to a group
consumer_one = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
                             client_id='consumer_one',
                             group_id='my_group',                       # the same group will be used by all consumers
                             auto_offset_reset='earliest',
                             consumer_timeout_ms=10000)

In [19]:
# subscribe `consumer_one` to the partitioned topic
consumer_one.subscribe(['a_partitioned_topic'])

---

### Consuming messages as a group
Use the `ConsumerGroup` notebook to:
1. create a second consumer `consumer_two`
2. assign it to the same consumer group `my_group`
3. subscribe to the same topic `a_partitioned_topic`

Each consumer within a group is going to be an independent process (should be run in parallel from the others) and will provide access to a fraction of the incoming data

In [20]:
# check the partitions assigned to `consumer_one`
consumer_one.assignment()

set()

Start the two consumers and process data in parallel from the two partitions.

Typically, one would achieve this by running the two consumers in separate threads, processes, or executors, depending on the desired computing architecture.

In [22]:
# use multiple consumers in parallel (`consumer_one` in this notebook)
for message in consumer_one:
    print(f"{message.partition}[{message.offset}]\t {json.loads(message.value)}")

0[17]	 {'name': 'John', 'surname': 'Johnson', 'amount': 868.9, 'delta_t': 8.3, 'flag': 0}
0[18]	 {'name': 'John', 'surname': 'Smith', 'amount': 116.84, 'delta_t': 8.58, 'flag': 0}
0[19]	 {'name': 'Joe', 'surname': 'Johnson', 'amount': 845.18, 'delta_t': 5.02, 'flag': 0}
0[20]	 {'name': 'Andy', 'surname': 'Jones', 'amount': 716.21, 'delta_t': 6.6, 'flag': 1}
0[21]	 {'name': 'Joe', 'surname': 'Smith', 'amount': 504.5, 'delta_t': 9.26, 'flag': 0}


In [23]:
# check the partitions assigned to `consumer_one`
consumer_one.assignment()

{TopicPartition(topic='a_partitioned_topic', partition=0)}

---

## Reading from the Kafka+Spark results topic

Let's subscribe to the `results` topic and monitor the frauds

In [27]:
consumer.subscribe(['results'])

for message in consumer:
    print ("%d[%d] k=%s v=%s" % (message.partition,
                          message.offset,
                          message.key,
                          message.value))
    #print ('      --> sending alert message to user {}\n'.format(message.key.decode('ascii')))

0[20] k=None v=b'{"name": "Andy", "surname": "Jones", "amount": 716.21, "delta_t": 6.6, "flag": 1}'
0[21] k=None v=b'{"name": "Joe", "surname": "Smith", "amount": 504.5, "delta_t": 9.26, "flag": 0}'
0[22] k=None v=b'{"name": "Andy", "surname": "Jones", "amount": 222.98, "delta_t": 4.21, "flag": 0}'
0[23] k=None v=b'{"name": "John", "surname": "Millers", "amount": 524.53, "delta_t": 5.01, "flag": 0}'
0[24] k=None v=b'{"name": "Andy", "surname": "Johnson", "amount": 673.56, "delta_t": 7.28, "flag": 0}'
0[25] k=None v=b'{"name": "John", "surname": "Johnson", "amount": 352.61, "delta_t": 1.54, "flag": 0}'
0[26] k=None v=b'{"name": "Andy", "surname": "Smith", "amount": 672.47, "delta_t": 2.17, "flag": 0}'
0[27] k=None v=b'{"name": "Joe", "surname": "Smith", "amount": 613.03, "delta_t": 3.15, "flag": 1}'
0[28] k=None v=b'{"name": "Joe", "surname": "Millers", "amount": 817.84, "delta_t": 3.46, "flag": 0}'
0[29] k=None v=b'{"name": "John", "surname": "Johnson", "amount": 376.6, "delta_t": 4.55